In [ ]:
# ── Cell 1 of Global_Server.ipynb ────────────────────────────
import sys, os, pickle, ssl, shutil, threading
import tenseal as ts
from flask import Flask, request

_HERE         = os.path.abspath(os.getcwd())
_PROJECT_ROOT = os.path.dirname(_HERE)
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)

from config import (
    RESULTS_PATH, SERVER_PORT, CERT_FILE, KEY_FILE,
    CHUNKS_FOLDER, AGG_CHUNKS_FOLDER, AGGREGATED_GRAD_FILE,
    N_CLIENTS
)

# ── Load CKKS context ─────────────────────────────────────────
CONTEXT_FILE = os.path.join(RESULTS_PATH, "context.ser")
os.makedirs(CHUNKS_FOLDER, exist_ok=True)

client_context = None
if os.path.exists(CONTEXT_FILE):
    with open(CONTEXT_FILE, "rb") as f:
        client_context = ts.context_from(f.read())
    print(f"[SERVER] CKKS context loaded from {CONTEXT_FILE}")
else:
    print(f"[SERVER] WARNING: context.ser not found — will load when first client posts")

# ── Track received chunks per round ───────────────────────────
# received_clients: set of client_ids that have posted ALL their chunks this round
received_clients = set()
CHUNKS_PER_CLIENT = 2   # gradient vector 13573 → 2 chunks at chunk_size=8000
client_chunk_counts = {}  # client_id → number of chunks received this round
lock = threading.Lock()

def reload_context_if_needed():
    global client_context
    if client_context is None and os.path.exists(CONTEXT_FILE):
        with open(CONTEXT_FILE, "rb") as f:
            client_context = ts.context_from(f.read())
        print(f"[SERVER] CKKS context loaded from {CONTEXT_FILE}")

def try_aggregate():
    """Called after every chunk receipt. Aggregates when all clients are ready."""
    global received_clients, client_chunk_counts

    if len(received_clients) < N_CLIENTS:
        return  # Not all clients posted yet

    print(f"[SERVER] All {N_CLIENTS} clients posted — starting ciphertext aggregation...")

    try:
        # Discover client directories
        client_dirs = [
            d for d in os.listdir(CHUNKS_FOLDER)
            if os.path.isdir(os.path.join(CHUNKS_FOLDER, d))
        ]

        # Load encrypted chunks per client
        chunk_groups = {}
        for client_id in client_dirs:
            client_path = os.path.join(CHUNKS_FOLDER, client_id)
            chunk_files = sorted(
                [f for f in os.listdir(client_path) if f.endswith(".bin")],
                key=lambda x: int(x.split("_")[-1].split(".")[0])
            )
            for fname in chunk_files:
                idx = int(fname.split("_")[-1].split(".")[0])
                with open(os.path.join(client_path, fname), "rb") as f:
                    payload = pickle.load(f)
                if isinstance(payload, dict) and "data" in payload:
                    vec = ts.ckks_vector_from(client_context, payload["data"])
                    chunk_groups.setdefault(idx, []).append(vec)
                    print(f"[AGG] Loaded {client_id}/{fname}")

        if not chunk_groups:
            print("[AGG] No valid chunks found — skipping aggregation")
            return

        # Homomorphic FedAvg in ciphertext space
        aggregated_chunks = []
        for idx, vectors in sorted(chunk_groups.items()):
            enc_sum = vectors[0]
            for vec in vectors[1:]:
                enc_sum = enc_sum + vec
            enc_avg = enc_sum * (1.0 / len(vectors))
            aggregated_chunks.append((idx, enc_avg))
            print(f"[AGG] Aggregated chunk {idx} ({len(vectors)} clients)")

        # Save aggregated chunks for audit trail
        os.makedirs(AGG_CHUNKS_FOLDER, exist_ok=True)
        for idx, enc_chunk in aggregated_chunks:
            path = os.path.join(AGG_CHUNKS_FOLDER, f"agg_chunk_{idx}.bin")
            with open(path, "wb") as f:
                pickle.dump(enc_chunk.serialize(), f)

        # Write the single file that all clients poll for
        all_serialized = [chunk.serialize() for _, chunk in aggregated_chunks]
        with open(AGGREGATED_GRAD_FILE, "wb") as f:
            pickle.dump(all_serialized, f)
        print(f"[AGG] ✓ Global gradient written → {AGGREGATED_GRAD_FILE}")

        # Clean up received chunks for next round
        shutil.rmtree(CHUNKS_FOLDER)
        os.makedirs(CHUNKS_FOLDER, exist_ok=True)
        print(f"[AGG] Cleaned up {CHUNKS_FOLDER}")

        # Reset tracking for next round
        received_clients.clear()
        client_chunk_counts.clear()
        print("[SERVER] Ready for next round\n")

    except Exception as e:
        print(f"[AGG] Aggregation failed: {e}")
        import traceback
        traceback.print_exc()

# ── Flask App ─────────────────────────────────────────────────
app = Flask(__name__)

@app.route("/", methods=["POST"])
def receive_chunk():
    global received_clients, client_chunk_counts

    reload_context_if_needed()
    if client_context is None:
        return "Context not ready", 503

    try:
        payload   = pickle.loads(request.data)
        chunk_id  = payload["chunk_id"]
        client_id = payload["client_id"]
        data      = payload["data"]

        # Validate CKKS ciphertext
        try:
            encrypted = ts.ckks_vector_from(client_context, data)
            if encrypted.size() < 5 or encrypted.size() > 10000:
                print(f"[SERVER] Dropped chunk {chunk_id} — invalid size: {encrypted.size()}")
                return "Corrupted", 400
        except Exception as e:
            print(f"[SERVER] CKKS validation failed for chunk {chunk_id} from {client_id}: {e}")
            return "Corrupted", 400

        # Save validated chunk
        client_folder = os.path.join(CHUNKS_FOLDER, client_id)
        os.makedirs(client_folder, exist_ok=True)
        filename = os.path.join(client_folder, f"chunk_{chunk_id}.bin")
        with open(filename, "wb") as f:
            pickle.dump({"data": data}, f)
        print(f"[SERVER] ✓ Received chunk {chunk_id} from {client_id}")

        # Track how many chunks this client has sent
        with lock:
            client_chunk_counts[client_id] = client_chunk_counts.get(client_id, 0) + 1
            if client_chunk_counts[client_id] >= CHUNKS_PER_CLIENT:
                received_clients.add(client_id)
                print(f"[SERVER] {client_id} complete ({len(received_clients)}/{N_CLIENTS} clients ready)")
            current_ready = len(received_clients)

        # Trigger aggregation in background thread when all clients ready
        if current_ready >= N_CLIENTS:
            t = threading.Thread(target=try_aggregate, daemon=True)
            t.start()

        return "OK", 200

    except Exception as e:
        print(f"[SERVER] Error: {e}")
        return "Error", 500

@app.route("/health", methods=["GET"])
def health():
    return f"OK — clients ready this round: {len(received_clients)}/{N_CLIENTS}", 200

# ── Start Server ──────────────────────────────────────────────
def run_server():
    if not os.path.exists(CERT_FILE):
        print(f"[SERVER] ERROR: cert.pem not found at {CERT_FILE}")
        return

    ssl_ctx = ssl.SSLContext(ssl.PROTOCOL_TLS_SERVER)
    ssl_ctx.load_cert_chain(certfile=CERT_FILE, keyfile=KEY_FILE)

    print(f"[SERVER] N_CLIENTS expected per round : {N_CLIENTS}")
    print(f"[SERVER] Chunks per client            : {CHUNKS_PER_CLIENT}")
    print(f"[SERVER] Chunks folder                : {CHUNKS_FOLDER}")
    print(f"[SERVER] Aggregated gradient file     : {AGGREGATED_GRAD_FILE}")
    print(f"[SERVER] Listening on https://127.0.0.1:{SERVER_PORT} ...")
    app.run(host="127.0.0.1", port=SERVER_PORT, ssl_context=ssl_ctx, threaded=True)

run_server()


[SERVER] WARNING: context.ser not found — will load when first client posts
[SERVER] N_CLIENTS expected per round : 3
[SERVER] Chunks per client            : 2
[SERVER] Chunks folder                : /Users/ravi/Desktop/Paper/hybrid-privacy-fl-healthcare/server/received_chunks_bin
[SERVER] Aggregated gradient file     : /Users/ravi/Desktop/Paper/hybrid-privacy-fl-healthcare/aggregated_gradient_global_encrypted.pkl
[SERVER] Listening on https://127.0.0.1:5055 ...
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on https://127.0.0.1:5055
Press CTRL+C to quit
127.0.0.1 - - [30/Jun/2026 17:49:52] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [30/Jun/2026 17:49:52] "POST / HTTP/1.1" 200 -


[SERVER] CKKS context loaded from /Users/ravi/Desktop/Paper/hybrid-privacy-fl-healthcare/results/context.ser
[SERVER] ✓ Received chunk 0 from hospital_1
[SERVER] ✓ Received chunk 1 from hospital_1
[SERVER] hospital_1 complete (1/3 clients ready)


127.0.0.1 - - [30/Jun/2026 17:49:54] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [30/Jun/2026 17:49:54] "POST / HTTP/1.1" 200 -


[SERVER] ✓ Received chunk 0 from hospital_2
[SERVER] ✓ Received chunk 1 from hospital_2
[SERVER] hospital_2 complete (2/3 clients ready)


...

[SERVER] ✓ Received chunk 0 from hospital_1
[SERVER] ✓ Received chunk 1 from hospital_1
[SERVER] hospital_1 complete (2/3 clients ready)
[SERVER] ✓ Received chunk 0 from hospital_3
[SERVER] ✓ Received chunk 1 from hospital_3
[SERVER] hospital_3 complete (3/3 clients ready)
[SERVER] All 3 clients posted — starting ciphertext aggregation...
[AGG] Loaded hospital_1/chunk_0.bin
[AGG] Loaded hospital_1/chunk_1.bin
[AGG] Loaded hospital_2/chunk_0.bin
[AGG] Loaded hospital_2/chunk_1.bin
[AGG] Loaded hospital_3/chunk_0.bin
[AGG] Loaded hospital_3/chunk_1.bin
[AGG] Aggregated chunk 0 (3 clients)
[AGG] Aggregated chunk 1 (3 clients)
[AGG] ✓ Global gradient written → /Users/ravi/Desktop/Paper/hybrid-privacy-fl-healthcare/aggregated_gradient_global_encrypted.pkl
[AGG] Cleaned up /Users/ravi/Desktop/Paper/hybrid-privacy-fl-healthcare/server/received_chunks_bin
[SERVER] Ready for next round



127.0.0.1 - - [30/Jun/2026 17:50:24] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [30/Jun/2026 17:50:24] "POST / HTTP/1.1" 200 -


[SERVER] ✓ Received chunk 0 from hospital_2
[SERVER] ✓ Received chunk 1 from hospital_2
[SERVER] hospital_2 complete (1/3 clients ready)


127.0.0.1 - - [30/Jun/2026 17:50:25] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [30/Jun/2026 17:50:25] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [30/Jun/2026 17:50:25] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [30/Jun/2026 17:50:25] "POST / HTTP/1.1" 200 -


[SERVER] ✓ Received chunk 0 from hospital_1
[SERVER] ✓ Received chunk 0 from hospital_3
[SERVER] ✓ Received chunk 1 from hospital_1
[SERVER] hospital_1 complete (2/3 clients ready)
[SERVER] ✓ Received chunk 1 from hospital_3
[SERVER] hospital_3 complete (3/3 clients ready)
[SERVER] All 3 clients posted — starting ciphertext aggregation...
[AGG] Loaded hospital_1/chunk_0.bin
[AGG] Loaded hospital_1/chunk_1.bin
[AGG] Loaded hospital_2/chunk_0.bin
[AGG] Loaded hospital_2/chunk_1.bin
[AGG] Loaded hospital_3/chunk_0.bin
[AGG] Loaded hospital_3/chunk_1.bin
[AGG] Aggregated chunk 0 (3 clients)
[AGG] Aggregated chunk 1 (3 clients)
[AGG] ✓ Global gradient written → /Users/ravi/Desktop/Paper/hybrid-privacy-fl-healthcare/aggregated_gradient_global_encrypted.pkl
[AGG] Cleaned up /Users/ravi/Desktop/Paper/hybrid-privacy-fl-healthcare/server/received_chunks_bin
[SERVER] Ready for next round

